# Profile history: how RentFriends profiles change over time

Scope: the `profile_history` table. One row is one version of one user's profile. Version 1 is the
creation, and every later row is an update. Each row holds a full snapshot of the profile at that
version, plus the list of fields that changed compared with the previous version.

Privacy: `bio` text is **not displayed**. Only `bio_length` and the fact that `bio` changed are shown.
The bio audit is still pending (see `docs/RENTFRIENDS.md`, section 4).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", None)

# One hue for magnitude; a lighter tint only for a second series. Grid stays recessive.
PRIMARY = "#2F5D8C"
LIGHT = "#9DB8D6"
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.6,
        "figure.dpi": 110,
    }
)

DATA_DIR = Path("../data/processed")
history = (
    pd.read_parquet(DATA_DIR / "profile_history.parquet")
    .sort_values(["user_key", "version"])
    .reset_index(drop=True)
)
users = pd.read_parquet(DATA_DIR / "users.parquet")

# Profile fields that are snapshotted in every version (bio is handled separately).
PROFILE_FIELDS = [
    "sex",
    "age_band",
    "city",
    "daily_routine",
    "lifestyle",
    "work_from_home",
    "cleanliness",
    "noise_level",
    "smoking",
    "alcohol",
    "interests",
    "budget",
    "room_type",
    "search_goal",
    "move_in_month_offset",
]

print(f"profile_history: {len(history)} rows, {history['user_key'].nunique()} users")

## 1. Structure of the table

| Column | Meaning |
|---|---|
| `user_key` | pseudonymised user id, same key as in `users` |
| `version` | sequence number per user, 1 = creation; unique per (`user_key`, `version`) |
| `change_type` | `created` (only for version 1) or `updated` |
| `day` | integer day index; the origin date is not documented in the repo |
| `changed_fields` | fields that differ from the previous version; empty for `created` |
| profile fields | full snapshot of the profile at that version (same meaning as in `users`) |
| `bio`, `bio_length` | bio snapshot; text not shown here |

The value `move_in` in `changed_fields` is a shortened name for `move_in_month_offset`. Section 5
checks this.

In [ ]:
overview = pd.DataFrame(
    {
        "value": [
            len(history),
            history["user_key"].nunique(),
            len(users) - history["user_key"].nunique(),
            history["version"].max(),
            (history["change_type"] == "created").sum(),
            (history["change_type"] == "updated").sum(),
        ]
    },
    index=[
        "rows (versions)",
        "users with history",
        "users in `users` with no history",
        "max version number",
        "created rows",
        "updated rows",
    ],
)
overview

## 2. Versions per user

A user without updates has only one version. The table shows how many users have each number of
versions.

In [ ]:
versions_per_user = history.groupby("user_key").size()
print(versions_per_user.value_counts().sort_index().to_string())
print(f"\nmedian versions per user: {versions_per_user.median()}")
print(f"users with 3 or more versions: {(versions_per_user >= 3).sum()}")

In [ ]:
distribution = versions_per_user.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(distribution.index.astype(str), distribution.values, color=PRIMARY)
ax.set_yscale("log")
ax.set_xlabel("versions per user")
ax.set_ylabel("users (log scale)")
ax.set_title("How many versions each user has")
plt.tight_layout()
plt.show()

## 3. Missingness in the snapshots

In [ ]:
snapshot_cols = PROFILE_FIELDS + ["bio_length"]
missing = pd.DataFrame(
    {
        "missing (rows)": history[snapshot_cols].isna().sum(),
        "missing %": (history[snapshot_cols].isna().mean() * 100).round(1),
    }
)
missing

In [ ]:
missing_pct = missing["missing %"].sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(missing_pct.index, missing_pct.values, color=PRIMARY)
for y, value in enumerate(missing_pct.values):
    ax.text(value + 1, y, f"{value:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 100)
ax.set_xlabel("missing (% of versions)")
ax.set_title("Share of missing values per field across all versions")
plt.tight_layout()
plt.show()

## 4. Value sets of categorical fields across all versions

Counts are per **row**, so a user with many versions is counted once per version. For per-user
counts, use `users` instead.

In [ ]:
CATEGORICAL = [
    "sex",
    "age_band",
    "city",
    "room_type",
    "search_goal",
    "daily_routine",
    "lifestyle",
    "work_from_home",
    "cleanliness",
    "noise_level",
    "smoking",
    "alcohol",
]

for col in CATEGORICAL:
    counts = history[col].value_counts(dropna=False)
    table = pd.DataFrame({"n": counts, "%": (counts / len(history) * 100).round(1)})
    print(f"\n### {col} ({len(counts)} values incl. NaN)")
    print(table.to_string())

## 5. What changes between versions

Counts of each field in `changed_fields` over all updates, and how many fields change in one update.

In [ ]:
updates = history[history["change_type"] == "updated"]

print("number of fields changed per update")
print(updates["changed_fields"].apply(len).value_counts().sort_index().to_string())

print("\nhow often each field appears in changed_fields")
changed_counts = pd.Series([f for fields in updates["changed_fields"] for f in fields])
print(changed_counts.value_counts().to_string())

In [ ]:
fields_per_update = updates["changed_fields"].apply(len).value_counts().sort_index()
field_counts = changed_counts.value_counts().sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
ax1.bar(fields_per_update.index.astype(str), fields_per_update.values, color=PRIMARY)
ax1.set_xlabel("fields changed in one update")
ax1.set_ylabel("updates")
ax1.set_title("Size of an update")
ax2.barh(field_counts.index, field_counts.values, color=PRIMARY)
ax2.set_xlabel("updates listing the field")
ax2.set_title("Which fields change most often (as listed)")
plt.tight_layout()
plt.show()

## 6. Does `changed_fields` match the snapshots?

For every update, compare the snapshot with the previous version. Then check whether the field is
listed in `changed_fields`. The agreement shows whether the list can be trusted, and whether
`move_in` means `move_in_month_offset`.

In [ ]:
LABEL = {"move_in_month_offset": "move_in"}  # name used in changed_fields


def differs_from_previous(col):
    previous = history.groupby("user_key")[col].shift()
    current = history[col]
    same = (current == previous) | (current.isna() & previous.isna())
    return ~same


is_update = history["change_type"] == "updated"
rows = []
for col in PROFILE_FIELDS:
    snapshot_changed = differs_from_previous(col)[is_update]
    label = LABEL.get(col, col)
    listed = pd.Series(
        [label in fields for fields in updates["changed_fields"]], index=updates.index
    )
    rows.append(
        {
            "field": col,
            "changed in snapshot": int(snapshot_changed.sum()),
            "listed in changed_fields": int(listed.sum()),
            "agreement %": round((snapshot_changed.values == listed.values).mean() * 100, 1),
        }
    )

pd.DataFrame(rows).set_index("field")

In [ ]:
agreement_table = pd.DataFrame(rows).set_index("field")[
    ["changed in snapshot", "listed in changed_fields"]
]
x = np.arange(len(agreement_table))
width = 0.4

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(
    x - width / 2,
    agreement_table["changed in snapshot"],
    width,
    color=PRIMARY,
    label="changed in snapshot",
)
ax.bar(
    x + width / 2,
    agreement_table["listed in changed_fields"],
    width,
    color=LIGHT,
    label="listed in changed_fields",
)
ax.set_xticks(x)
ax.set_xticklabels(agreement_table.index, rotation=45, ha="right")
ax.set_ylabel("updates")
ax.set_title("Snapshot changes vs. changed_fields, per field")
ax.legend()
plt.tight_layout()
plt.show()

Reading the agreement table:

- Most fields agree in 100% of updates, so `changed_fields` can be used directly for them.
- `sex` is never listed in `changed_fields`, although its snapshot changes in about 100 updates.
  The list is incomplete for this field, so compare snapshots when `sex` matters.
- `city` is listed more often (541) than its snapshot changes (303). Some listed changes have no
  visible change in the snapshot. Compare snapshots for `city` as well.
- `move_in` agrees in 99.4% of updates. This supports reading `move_in` as `move_in_month_offset`.

## 7. Timing of versions

`day` is an integer index. Gaps between versions of the same user are in the same units.

In [ ]:
history["days_since_previous"] = history.groupby("user_key")["day"].diff()

print(f"day range: {history['day'].min()} to {history['day'].max()}")
print("\ndays since previous version (updated rows only)")
print(history.loc[is_update, "days_since_previous"].describe().round(1).to_string())

In [ ]:
gaps = history.loc[is_update, "days_since_previous"].dropna()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(gaps, bins=40, color=PRIMARY)
ax.set_xlabel("days since the previous version")
ax.set_ylabel("updates")
ax.set_title("Gap between consecutive versions of the same user")
plt.tight_layout()
plt.show()

## 8. Consistency with `users`

The latest version of each user should equal the current row in `users`. The comparison treats two
missing values as equal.

In [ ]:
latest = history.groupby("user_key").tail(1).set_index("user_key")
current = users.set_index("user_key")
shared = latest.index.intersection(current.index)

agreement = {}
for col in PROFILE_FIELDS:
    a = current.loc[shared, col]
    b = latest.loc[shared, col]
    agreement[col] = round(((a == b) | (a.isna() & b.isna())).mean() * 100, 1)

print(f"users compared: {len(shared)}")
pd.Series(agreement, name="agreement %").to_frame()

## 9. One user in three profile versions

The user is picked with `random_state=42` among users with at least three versions, where at least
three profile fields take different values across versions. This keeps the three shown versions
visibly different. The three shown versions are the first (creation), a middle one and the
last one. Bio text is replaced by its length.

In [ ]:
multi = history[history["user_key"].isin(versions_per_user[versions_per_user >= 3].index)]
# Number of profile fields that take more than one value across the user's versions.
fields_varying = (
    multi.groupby("user_key")[PROFILE_FIELDS]
    .agg(lambda s: s.astype(str).nunique())
    .gt(1)
    .sum(axis=1)
)
eligible = fields_varying[fields_varying >= 3].index
user_id = pd.Series(eligible).sample(n=1, random_state=42).iloc[0]

versions = history[history["user_key"] == user_id].reset_index(drop=True)
picked_idx = [0, len(versions) // 2, len(versions) - 1]
picked = versions.iloc[picked_idx]

print(f"user_key: {user_id}")
print(f"total versions: {len(versions)}; shown: {[int(v) for v in picked['version']]}")

labels = [f"v{int(r.version)} | {r.change_type} | day {int(r.day)}" for r in picked.itertuples()]
view = picked[PROFILE_FIELDS + ["bio_length"]].T
view.columns = labels
view.loc["changed_fields (vs previous)"] = [
    ", ".join(fields) if len(fields) else "(creation)" for fields in picked["changed_fields"]
]
view

In [ ]:
# Which fields changed at each version of the example user (dark = changed vs previous version).
user_rows = history[history["user_key"] == user_id].reset_index(drop=True)


def changed_from_previous(col):
    current = user_rows[col]
    previous = current.shift()
    same = (current.astype(str) == previous.astype(str)) | (current.isna() & previous.isna())
    return (~same) & (user_rows.index > 0)


flags = pd.DataFrame({col: changed_from_previous(col) for col in PROFILE_FIELDS}).T.astype(int)
flags = flags[flags.sum(axis=1) > 0]  # show only fields that changed at least once

fig, ax = plt.subplots(figsize=(10, 0.4 * len(flags) + 1.5))
ax.imshow(flags.values, aspect="auto", cmap=ListedColormap(["#F2F2F2", PRIMARY]), vmin=0, vmax=1)
ax.set_yticks(range(len(flags.index)))
ax.set_yticklabels(flags.index)
ax.set_xticks(range(flags.shape[1]))
ax.set_xticklabels([f"v{v}" for v in user_rows["version"]])
ax.grid(False)
ax.set_title("Fields that changed in each version of the example user")
plt.tight_layout()
plt.show()

In [ ]:
def tag_set(raw):
    return set(json.loads(raw)) if raw is not None else set()


print("Interest tags added and removed between the shown versions")
for old, new in zip(picked.iloc[:-1].itertuples(), picked.iloc[1:].itertuples(), strict=False):
    old_tags, new_tags = tag_set(old.interests), tag_set(new.interests)
    print(f"\nv{int(old.version)} -> v{int(new.version)}")
    print("  added:  ", sorted(new_tags - old_tags) or "none")
    print("  removed:", sorted(old_tags - new_tags) or "none")

## 10. Bio changes (statistics only)

Bio text is **not displayed**. The patterns below are checked in memory, and only counts and
lengths are shown. They are a rough screen for personal data (digits, `@` or links, long digit
runs such as phone numbers), not a complete audit.

In [ ]:
import re

PATTERNS = {
    "contains digits": r"\d",
    "contains @ or link": r"@|https?://|www\.",
    "long digit run (9+ digits)": r"(?:\d\s?){9,}",
}


def bio_flags(text):
    if not isinstance(text, str):
        return {**{name: False for name in PATTERNS}, "empty": True}
    flags = {name: bool(re.search(pattern, text)) for name, pattern in PATTERNS.items()}
    flags["empty"] = len(text.strip()) == 0
    return flags


flags = pd.DataFrame([bio_flags(text) for text in history["bio"]], index=history.index)
bio_changed = history["changed_fields"].apply(lambda fields: "bio" in fields)
text_length_matches = (history["bio"].fillna("").str.len() == history["bio_length"]).mean()

bio_stats = pd.Series(
    {
        "versions": len(history),
        "versions where bio changed": int(bio_changed.sum()),
        "users who changed bio at least once": history.loc[bio_changed, "user_key"].nunique(),
        "empty bio, share of versions %": round(flags["empty"].mean() * 100, 1),
        "bio_length median (non-empty versions)": history.loc[
            history["bio_length"] > 0, "bio_length"
        ].median(),
        "bio_length maximum": history["bio_length"].max(),
        "bio_length equals text length, share %": round(text_length_matches * 100, 1),
    }
)
print(bio_stats.to_string())

print("\nshare of versions with each pattern (%)")
print((flags[list(PATTERNS)].mean() * 100).round(2).to_string())

In [ ]:
bio_changes_per_user = history[bio_changed].groupby("user_key").size()
candidates = bio_changes_per_user[bio_changes_per_user >= 2].index
bio_user = pd.Series(candidates).sample(n=1, random_state=42).iloc[0]

selected = history["user_key"] == bio_user
bio_table = pd.DataFrame(
    {
        "version": history.loc[selected, "version"].values,
        "change_type": history.loc[selected, "change_type"].values,
        "day": history.loc[selected, "day"].values,
        "bio changed": bio_changed[selected].values,
        "bio_length": history.loc[selected, "bio_length"].values,
        "empty": flags.loc[selected, "empty"].values,
        **{name: flags.loc[selected, name].values for name in PATTERNS},
    }
)

print(
    f"user_key: {bio_user}; versions: {len(bio_table)}; "
    f"versions with a bio change: {bio_table['bio changed'].sum()}"
)
bio_table.set_index("version")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.step(bio_table["version"], bio_table["bio_length"], where="post", color=PRIMARY, linewidth=2)
changed_points = bio_table[bio_table["bio changed"]]
ax1.scatter(
    changed_points["version"],
    changed_points["bio_length"],
    color=PRIMARY,
    s=40,
    zorder=3,
    label="version where bio changed",
)
ax1.set_xticks(bio_table["version"])
ax1.set_xlabel("version")
ax1.set_ylabel("bio length (characters)")
ax1.set_title("Example user: bio length by version")
ax1.legend()

non_empty = history.loc[history["bio_length"] > 0, "bio_length"]
ax2.hist(non_empty, bins=50, color=PRIMARY)
ax2.set_xlabel("bio length (characters)")
ax2.set_ylabel("versions")
ax2.set_title("Bio length, all non-empty versions")

plt.tight_layout()
plt.show()

## 11. Open points

- The origin of `day` is not documented. Relative timing within a user is clear, absolute dates are not.
- `changed_fields` agreement (section 6) decides whether the field list can be used directly or whether
  snapshots must be compared.
- Nine users in `users` have no history rows. Check whether these are profiles that never changed or
  missing records.
- The bio text is not analysed here. Its audit is still pending.